In [0]:
from pyspark import pipelines as dp

@dp.materialized_view(
    name="electroniz_product_sales_quarter_year",
    comment="Finance domain gold-layer materialized view that aggregates total product sales revenue by quarter and year across in-store and e-commerce channels. Combines data from silver-layer store orders (electroniz_silver_store_orders) and e-commerce transactions (electroniz_silver_ecomm_transactions) to provide a unified quarterly sales summary for financial reporting and trend analysis.",
    table_properties={
        "domain": "finance",
        "data_product_name": "total sales by quarter and year"
    },
    schema="""
        year INT COMMENT 'The calendar year extracted from the order date, representing the fiscal or calendar year in which the sales transactions occurred.',
        quarter INT COMMENT 'The calendar quarter (1-4) extracted from the order date, representing the fiscal quarter within the year for seasonal and periodic financial analysis.',
        total_sales DOUBLE COMMENT 'The cumulative gross sales revenue for the given quarter and year, aggregated from both in-store orders (electroniz_silver_store_orders) and e-commerce transactions (electroniz_silver_ecomm_transactions).'
    """
)
def product_sales_quarter_year():
    return spark.sql("""
        SELECT
            YEAR(order_date) AS year,
            QUARTER(order_date) AS quarter,
            SUM(sale_price) AS total_sales
        FROM (
            SELECT order_date, sale_price
            FROM electroniz_catalog.electroniz_silver_schema.electroniz_silver_store_orders
            UNION ALL
            SELECT order_date, sale_price
            FROM electroniz_catalog.electroniz_silver_schema.electroniz_silver_ecomm_transactions
        ) combined_sales
        GROUP BY YEAR(order_date), QUARTER(order_date)
        ORDER BY year, quarter
    """)